# b_validate

Run all cells. Outputs are written to this task's `output/` folder.


In [1]:
%run ~/Desktop/SHL_dblp_comparable/common/core.ipynb


/Users/slmagid/miniforge3/envs/research313/lib/python3.13/site-packages/nbformat/__init__.py:96: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
ROOT = Path.home() / 'Desktop' / 'SHL_dblp_comparable'
CONFIG = read_config(ROOT)
UP = ROOT / 'a_import' / 'output'
OUTPUT = ROOT / 'b_validate' / 'output'
OUTPUT.mkdir(parents=True, exist_ok=True)
raw = pd.read_csv(UP / 'df_traj_all.csv', low_memory=False)
person_col = 'dblp_id' if 'dblp_id' in raw else 'dblp' if 'dblp' in raw else None
age_col = 'CareerAgeZero' if 'CareerAgeZero' in raw else 'CareerAge' if 'CareerAge' in raw else None
q_col = 'pubs_adj' if 'pubs_adj' in raw else None
if None in (person_col, age_col, q_col):
    raise ValueError(f'Required person/age/pubs_adj columns absent; found {list(raw.columns)}')
raw[age_col] = pd.to_numeric(raw[age_col], errors='coerce')
raw[q_col] = pd.to_numeric(raw[q_col], errors='coerce')
raw = raw.dropna(subset=[person_col, age_col, q_col]).copy()
raw['source_person_id'] = raw[person_col].map(canonical_id)
if (raw[q_col] < 0).any():
    raise ValueError('Negative productivity in DBLP input')
if not np.allclose(raw[age_col], raw[age_col].round()):
    raise ValueError('Noninteger career ages')
raw[age_col] = raw[age_col].astype(int)
raw = raw[raw[age_col].ge(0)].copy()
duplicate_rows = int(raw.duplicated(['source_person_id', age_col]).sum())
panel_source = raw.groupby(['source_person_id', age_col], as_index=False)[q_col].sum().rename(columns={age_col: 'age', q_col: 'q'})
source_max = panel_source.groupby('source_person_id').age.max().rename('source_max_age')
has_zero = panel_source.groupby('source_person_id').age.min().eq(0)
eligible = source_max.index[has_zero.reindex(source_max.index, fill_value=False) & source_max.ge(1)]
excluded_no_zero = int((~has_zero.reindex(source_max.index, fill_value=False)).sum())
excluded_q0_only = int(source_max.eq(0).sum())
panel_source = panel_source[panel_source.source_person_id.isin(eligible)]
parts = []
imputed = 0
for person, g in panel_source.groupby('source_person_id', sort=False):
    max_age = min(int(g.age.max()), int(CONFIG['max_observed_years']) - 1)
    series = g.set_index('age').q.reindex(range(max_age + 1))
    imputed += int(series.isna().sum())
    series = series.fillna(0.0)
    parts.append(pd.DataFrame({'source_person_id': person, 'age': series.index, 'q': series.values}))
panel = pd.concat(parts, ignore_index=True).merge(source_max, on='source_person_id', how='left')
panel['split'] = panel.source_person_id.map(lambda x: split_for(x, CONFIG['split_seed']))
people = panel[['source_person_id', 'source_max_age', 'split']].drop_duplicates().copy()
people['all_eligible_cropped13'] = True
people['complete13_cropped'] = people.source_max_age.ge(12)
people['legacy_complete20_cropped13'] = people.source_max_age.ge(20)

panel.to_csv(OUTPUT / 'validated_panel.csv.gz', index=False, compression='gzip')
people.to_csv(OUTPUT / 'person_cohorts.csv', index=False)
report = {'status': 'passed', 'raw_rows': len(raw), 'eligible_people': len(people), 'duplicate_person_age_rows_aggregated': duplicate_rows, 'interior_missing_person_years_filled_zero': imputed, 'excluded_without_age0': excluded_no_zero, 'excluded_q0_only': excluded_q0_only, 'crop': 'ages 0..12', 'cohort_counts': {c: int(people[c].sum()) for c in people.columns if c not in {'source_person_id', 'source_max_age', 'split'}}, 'zero_fill_rationale': 'matches original DBLP panel pivot/fillna policy; AARC panels already contain explicit zero years'}
write_json(OUTPUT / 'quality_report.json', report)
print(pd.Series(report).to_string())


status                                                                                  passed
raw_rows                                                                                 29119
eligible_people                                                                           1963
duplicate_person_age_rows_aggregated                                                         0
interior_missing_person_years_filled_zero                                                    0
excluded_without_age0                                                                      119
excluded_q0_only                                                                             4
crop                                                                                ages 0..12
cohort_counts                                {'all_eligible_cropped13': 1963, 'complete13_c...
zero_fill_rationale                          matches original DBLP panel pivot/fillna polic...
